# Advanced Scratch Experiments

Three new from-scratch architectures:
- **SE-ResNet34** — deeper SE-ResNet18, channel attention
- **DenseNet-BC-100** — proper BC config (k=12, n=16, theta=0.5, ~0.8M params)
- **ScatteringResNet34** — fixed Gabor front-end (J=2, L=8) + SE-ResNet34 classifier

Training: **Mixup + CutMix** (1/3 each + 1/3 normal). Everything else identical to main notebook.

## 1. Setup

In [1]:
from pathlib import Path
import random
import time

import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from tqdm.auto import tqdm

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_DIR      = ROOT / 'data'
OUTPUT_DIR    = ROOT / 'outputs'
CHECKPOINT_DIR = OUTPUT_DIR / 'checkpoints'
SUBMISSION_DIR = OUTPUT_DIR / 'submissions'
FIGURE_DIR    = OUTPUT_DIR / 'figures'

for p in [CHECKPOINT_DIR, SUBMISSION_DIR, FIGURE_DIR]:
    p.mkdir(parents=True, exist_ok=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)
if torch.cuda.is_available():
    print('GPU   :', torch.cuda.get_device_name(0))
    print(f'VRAM  : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

Device: cuda
GPU   : NVIDIA RTXA6000-48Q
VRAM  : 51.3 GB


## 2. Load Metadata

In [2]:
train_csv = DATA_DIR / 'train.csv'
train_dir = DATA_DIR / 'train'
test_dir  = DATA_DIR / 'test'

df = pd.read_csv(train_csv, sep=';')
df['id']    = df['id'].astype(int)
df['label'] = df['label'].astype(int)
df['path']  = df['id'].map(lambda x: train_dir / f'{x}.tif')

print(df['label'].value_counts().sort_index())
print('Train images:', len(df))
print('Test images :', len(list(test_dir.glob('*.tif'))))

label
0    300
1    300
2    262
3    300
4    300
5    299
6    300
7    300
Name: count, dtype: int64
Train images: 2361
Test images : 789


## 3. Transforms + Dataset

In [3]:
IMG_SIZE    = (384, 576)   # height, width — preserves original 512:768 ratio
BATCH_SIZE  = 32
NUM_WORKERS = 0

train_tfms = transforms.Compose([
    transforms.Resize((432, 648)),
    transforms.RandomCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=5),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5]),
    transforms.RandomErasing(p=0.10, scale=(0.01, 0.04), ratio=(0.3, 3.3)),
])

eval_tfms = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5]),
])


class TildaDataset(Dataset):
    def __init__(self, dataframe=None, image_dir=None, ids=None, transform=None, has_labels=True):
        self.dataframe  = dataframe.reset_index(drop=True) if dataframe is not None else None
        self.image_dir  = Path(image_dir) if image_dir is not None else None
        self.ids        = list(ids) if ids is not None else None
        self.transform  = transform
        self.has_labels = has_labels

    def __len__(self):
        return len(self.dataframe) if self.has_labels else len(self.ids)

    def __getitem__(self, idx):
        if self.has_labels:
            row      = self.dataframe.iloc[idx]
            image_id = int(row['id'])
            path     = Path(row['path'])
            label    = int(row['label'])
        else:
            image_id = int(self.ids[idx])
            path     = self.image_dir / f'{image_id}.tif'
            label    = -1
        image = Image.open(path).convert('L')
        if self.transform is not None:
            image = self.transform(image)
        return image, label, image_id


test_ids    = sorted([int(p.stem) for p in test_dir.glob('*.tif')])
test_ds     = TildaDataset(image_dir=test_dir, ids=test_ids, transform=eval_tfms, has_labels=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=True)
print('Test loader ready, batches:', len(test_loader))

Test loader ready, batches: 25


## 4. Model Definitions

### SE-ResNet34
Exact same SE-ResNet backbone as SE-ResNet18 already tested, with the ResNet-34 schedule `[3,4,6,3]`.

### DenseNet-BC-100
Proper Bottleneck + Compression DenseNet: k=12, 16 layers per block × 3 blocks, theta=0.5. ~0.8M params.

### Scattering Network + SE-ResNet34
Fixed Gabor filter bank (J=2 scales, L=8 orientations) as a non-trainable front-end, followed by SE-ResNet34.
The Gabor filters replace the first conv layers — they are mathematically designed for texture and require **no learning**.
After the filter bank (4× spatial downsampling): 17-channel feature maps at 96×144.

In [4]:
# ═══════════════════════════════════════════════════════════════════════
# SE Block
# ═══════════════════════════════════════════════════════════════════════

class SEBlock(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        hidden = max(channels // reduction, 8)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc   = nn.Sequential(
            nn.Linear(channels, hidden),
            nn.ReLU(inplace=True),
            nn.Linear(hidden, channels),
            nn.Sigmoid(),
        )

    def forward(self, x):
        b, c, _, _ = x.shape
        s = self.pool(x).view(b, c)
        return x * self.fc(s).view(b, c, 1, 1)


# ═══════════════════════════════════════════════════════════════════════
# SE-ResNet basic block
# ═══════════════════════════════════════════════════════════════════════

class SEResNetBasicBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1, reduction=16):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(out_ch)
        self.se    = SEBlock(out_ch, reduction=reduction)
        self.relu  = nn.ReLU(inplace=True)
        self.down  = None
        if stride != 1 or in_ch != out_ch:
            self.down = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_ch),
            )

    def forward(self, x):
        identity = x if self.down is None else self.down(x)
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return self.relu(self.se(out) + identity)


# ═══════════════════════════════════════════════════════════════════════
# SE-ResNet backbone (variable in_channels and layer schedule)
# ═══════════════════════════════════════════════════════════════════════

class SEResNetScratch(nn.Module):
    def __init__(self, layers, in_channels=1, stem_stride=2, widths=(64, 128, 256, 512), num_classes=8):
        super().__init__()
        self.in_ch = widths[0]
        self.stem  = nn.Sequential(
            nn.Conv2d(in_channels, widths[0], 7, stride=stem_stride, padding=3, bias=False),
            nn.BatchNorm2d(widths[0]), nn.ReLU(inplace=True),
            nn.MaxPool2d(3, stride=2, padding=1),
        )
        self.layer1 = self._make(widths[0], layers[0], stride=1)
        self.layer2 = self._make(widths[1], layers[1], stride=2)
        self.layer3 = self._make(widths[2], layers[2], stride=2)
        self.layer4 = self._make(widths[3], layers[3], stride=2)
        self.head   = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Dropout(0.30), nn.Linear(widths[3], num_classes),
        )

    def _make(self, out_ch, n, stride):
        layers = [SEResNetBasicBlock(self.in_ch, out_ch, stride=stride)]
        self.in_ch = out_ch
        for _ in range(1, n):
            layers.append(SEResNetBasicBlock(self.in_ch, out_ch))
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x); x = self.layer2(x)
        x = self.layer3(x); x = self.layer4(x)
        return self.head(x)


class SEResNet34Scratch(SEResNetScratch):
    def __init__(self, num_classes=8):
        super().__init__(layers=(3, 4, 6, 3), in_channels=1, num_classes=num_classes)


# ═══════════════════════════════════════════════════════════════════════
# DenseNet-BC-100  (k=12, n=16 per block, 3 blocks, theta=0.5)
# ═══════════════════════════════════════════════════════════════════════

class _DenseLayer(nn.Module):
    def __init__(self, in_ch, k, dropout=0.10):
        super().__init__()
        self.net = nn.Sequential(
            nn.BatchNorm2d(in_ch), nn.ReLU(inplace=True),
            nn.Conv2d(in_ch, 4*k, 1, bias=False),
            nn.BatchNorm2d(4*k), nn.ReLU(inplace=True),
            nn.Conv2d(4*k, k, 3, padding=1, bias=False),
            nn.Dropout2d(dropout),
        )
    def forward(self, x):
        return torch.cat([x, self.net(x)], dim=1)


class _DenseBlock(nn.Module):
    def __init__(self, in_ch, n, k, dropout=0.10):
        super().__init__()
        ch, layers = in_ch, []
        for _ in range(n):
            layers.append(_DenseLayer(ch, k, dropout)); ch += k
        self.net = nn.Sequential(*layers)
        self.out_channels = ch
    def forward(self, x):
        return self.net(x)


class _Transition(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.BatchNorm2d(in_ch), nn.ReLU(inplace=True),
            nn.Conv2d(in_ch, out_ch, 1, bias=False),
            nn.AvgPool2d(2),
        )
    def forward(self, x):
        return self.net(x)


class DenseNetBC100(nn.Module):
    def __init__(self, k=12, n_per_block=16, theta=0.5, num_classes=8):
        super().__init__()
        ch = 2 * k
        self.stem = nn.Sequential(
            nn.Conv2d(1, ch, 7, stride=2, padding=3, bias=False),
            nn.BatchNorm2d(ch), nn.ReLU(inplace=True),
            nn.MaxPool2d(3, stride=2, padding=1),
        )
        blocks = []
        for i in range(3):
            b = _DenseBlock(ch, n_per_block, k)
            blocks.append(b); ch = b.out_channels
            if i < 2:
                out_ch = int(ch * theta)
                blocks.append(_Transition(ch, out_ch)); ch = out_ch
        self.features = nn.Sequential(*blocks)
        self.head = nn.Sequential(
            nn.BatchNorm2d(ch), nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Dropout(0.25), nn.Linear(ch, num_classes),
        )

    def forward(self, x):
        return self.head(self.features(self.stem(x)))


# ═══════════════════════════════════════════════════════════════════════
# Scattering Network  (fixed Gabor front-end + SE-ResNet34 classifier)
# ═══════════════════════════════════════════════════════════════════════

class GaborScattering(nn.Module):
    """
    First-order scattering using fixed Gabor filters.
    J scales x L orientations + Gaussian lowpass.
    Weights are NOT trained — mathematically designed for texture.

    Output shape: (B, 1 + J*L, H/2^J, W/2^J)
    For J=2, L=8 and input 384x576: (B, 17, 96, 144)
    """
    def __init__(self, J=2, L=8, filter_size=31):
        super().__init__()
        self.J     = J
        self.L     = L
        self.n_out = 1 + J * L

        filters = []
        for j in range(J):
            scale = 2 ** j
            for l in range(L):
                theta = l * np.pi / L
                f = self._gabor(filter_size, sigma=2.5*scale, theta=theta,
                                Lambda=5.0*scale, gamma=0.5)
                filters.append(f)
        filters.append(self._gaussian(filter_size, sigma=2.5 * 2**J))

        kernels = torch.FloatTensor(np.stack(filters)).unsqueeze(1)  # (J*L+1, 1, F, F)
        self.register_buffer('kernels', kernels)
        self.pad = filter_size // 2

    @staticmethod
    def _gabor(size, sigma, theta, Lambda, gamma):
        half = size // 2
        y, x = np.mgrid[-half:half+1, -half:half+1]
        y, x = y[:size, :size].astype(float), x[:size, :size].astype(float)
        x_t =  x * np.cos(theta) + y * np.sin(theta)
        y_t = -x * np.sin(theta) + y * np.cos(theta)
        gb  = np.exp(-0.5 * (x_t**2 + (gamma*y_t)**2) / sigma**2)
        gb *= np.cos(2 * np.pi * x_t / Lambda)
        gb -= gb.mean()
        return (gb / (np.sqrt((gb**2).sum()) + 1e-8)).astype(np.float32)

    @staticmethod
    def _gaussian(size, sigma):
        half = size // 2
        y, x = np.mgrid[-half:half+1, -half:half+1]
        y, x = y[:size, :size].astype(float), x[:size, :size].astype(float)
        g = np.exp(-(x**2 + y**2) / (2 * sigma**2))
        return (g / g.sum()).astype(np.float32)

    def forward(self, x):
        out    = F.conv2d(x, self.kernels, padding=self.pad)  # (B, J*L+1, H, W)
        gabors = torch.abs(out[:, :-1])    # modulus of wavelet responses
        lp     = out[:, -1:]               # lowpass (no modulus needed)
        feat   = torch.cat([lp, gabors], dim=1)
        return F.avg_pool2d(feat, 2**self.J)  # 4x spatial reduction for J=2


class ScatteringResNet34(nn.Module):
    """
    Fixed Gabor scattering (J=2, L=8) + SE-ResNet34 classifier.

    After scattering: (B, 17, 96, 144)  — no trainable params in this part
    Classifier stem: 3x3 conv (no stride) preserving spatial detail
    Then 4 residual stages with SE attention
    """
    def __init__(self, J=2, L=8, num_classes=8):
        super().__init__()
        self.scatter = GaborScattering(J=J, L=L)
        n_in = self.scatter.n_out   # 17 for J=2, L=8

        # Classifier: SE-ResNet34 with n_in input channels
        # Use stem_stride=1 — spatial dims already reduced by scattering
        self.backbone = SEResNetScratch(
            layers=(3, 4, 6, 3),
            in_channels=n_in,
            stem_stride=1,
            num_classes=num_classes,
        )

    def forward(self, x):
        return self.backbone(self.scatter(x))


# ═══════════════════════════════════════════════════════════════════════
# Registry
# ═══════════════════════════════════════════════════════════════════════

MODEL_REGISTRY = {
    'se_resnet34_scratch': SEResNet34Scratch,
    'densenet_bc100':      DenseNetBC100,
    'scatter_resnet34':    ScatteringResNet34,
}


def build_model(name, num_classes=8):
    if name not in MODEL_REGISTRY:
        raise ValueError(f'Unknown model: {name}')
    return MODEL_REGISTRY[name](num_classes=num_classes)


for name in MODEL_REGISTRY:
    m = build_model(name).to(DEVICE)
    total = sum(p.numel() for p in m.parameters())
    fixed = sum(p.numel() for p in m.parameters() if not p.requires_grad)
    print(f'{name:<25}: {total:>12,} params  ({fixed:,} fixed)')
    del m
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()

se_resnet34_scratch      :   21,445,248 params  (0 fixed)
densenet_bc100           :      769,052 params  (0 fixed)
scatter_resnet34         :   21,495,424 params  (0 fixed)


## 5. Mixup / CutMix

In [5]:
def mixup_batch(images, labels, alpha=0.4):
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(images.size(0), device=images.device)
    return lam * images + (1 - lam) * images[idx], labels, labels[idx], lam


def cutmix_batch(images, labels, alpha=1.0):
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(images.size(0), device=images.device)
    _, _, H, W = images.shape
    cut_w = int(W * np.sqrt(1 - lam))
    cut_h = int(H * np.sqrt(1 - lam))
    cx, cy = np.random.randint(W), np.random.randint(H)
    x1 = np.clip(cx - cut_w // 2, 0, W)
    x2 = np.clip(cx + cut_w // 2, 0, W)
    y1 = np.clip(cy - cut_h // 2, 0, H)
    y2 = np.clip(cy + cut_h // 2, 0, H)
    mixed = images.clone()
    mixed[:, :, y1:y2, x1:x2] = images[idx, :, y1:y2, x1:x2]
    lam = 1 - (x2 - x1) * (y2 - y1) / (W * H)
    return mixed, labels, labels[idx], lam


def mix_loss(criterion, logits, ya, yb, lam):
    return lam * criterion(logits, ya) + (1 - lam) * criterion(logits, yb)

## 6. Training Loop

In [6]:
def run_epoch(model, loader, criterion, optimizer=None):
    is_train = optimizer is not None
    model.train(is_train)
    total_loss, all_preds, all_labels = 0.0, [], []
    with torch.set_grad_enabled(is_train):
        for images, labels, _ in tqdm(loader, leave=False):
            images = images.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)
            logits = model(images)
            loss   = criterion(logits, labels)
            if is_train:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
                optimizer.step()
            total_loss += loss.item() * images.size(0)
            all_preds.extend(logits.argmax(1).detach().cpu().numpy())
            all_labels.extend(labels.detach().cpu().numpy())
    return total_loss / len(loader.dataset), accuracy_score(all_labels, all_preds)


def run_epoch_mix(model, loader, criterion, optimizer):
    model.train()
    total_loss, all_preds, all_labels = 0.0, [], []
    for images, labels, _ in tqdm(loader, leave=False):
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        r = random.random()
        if r < 0.33:
            mixed, ya, yb, lam = mixup_batch(images, labels)
            logits = model(mixed)
            loss   = mix_loss(criterion, logits, ya, yb, lam)
        elif r < 0.66:
            mixed, ya, yb, lam = cutmix_batch(images, labels)
            logits = model(mixed)
            loss   = mix_loss(criterion, logits, ya, yb, lam)
        else:
            logits = model(images)
            loss   = criterion(logits, labels)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()
        total_loss += loss.item() * images.size(0)
        all_preds.extend(logits.argmax(1).detach().cpu().numpy())
        all_labels.extend(labels.detach().cpu().numpy())
    return total_loss / len(loader.dataset), accuracy_score(all_labels, all_preds)


def train_model(model, train_loader, val_loader, model_name,
                epochs=200, lr=0.03, weight_decay=5e-4, patience=50):
    criterion  = nn.CrossEntropyLoss(label_smoothing=0.05)
    optimizer  = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9,
                                  weight_decay=weight_decay, nesterov=True)
    scheduler  = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-5)
    history    = []
    best_acc   = -1.0
    best_epoch = 0
    best_path  = CHECKPOINT_DIR / f'best_{model_name}.pt'
    start      = time.time()

    for epoch in range(1, epochs + 1):
        tr_loss, tr_acc = run_epoch_mix(model, train_loader, criterion, optimizer)
        va_loss, va_acc = run_epoch(model, val_loader, criterion, optimizer=None)
        scheduler.step()
        history.append({'model': model_name, 'epoch': epoch,
                        'train_loss': tr_loss, 'train_acc': tr_acc,
                        'val_loss': va_loss,   'val_acc': va_acc,
                        'lr': scheduler.get_last_lr()[0]})
        if va_acc > best_acc:
            best_acc = va_acc; best_epoch = epoch
            torch.save({'model_name': model_name, 'model_state_dict': model.state_dict(),
                        'best_acc': best_acc, 'best_epoch': best_epoch, 'img_size': IMG_SIZE}, best_path)
        print(f'{model_name} | ep {epoch:03d} | train {tr_acc:.4f}/{tr_loss:.4f} | '
              f'val {va_acc:.4f}/{va_loss:.4f} | best {best_acc:.4f} @ {best_epoch}')
        if epoch - best_epoch >= patience:
            print(f'{model_name}: early stop — no improvement for {patience} epochs.')
            break

    elapsed = (time.time() - start) / 60
    print(f'{model_name}: {elapsed:.1f} min — best val acc {best_acc:.4f}')
    return pd.DataFrame(history), best_path, best_acc, best_epoch, elapsed

## 7. TTA Inference

In [7]:
def predict_proba(model, loader, use_tta=True):
    model.eval()
    all_probs, all_ids = [], []
    with torch.no_grad():
        for images, _, image_ids in tqdm(loader, leave=False):
            images = images.to(DEVICE, non_blocking=True)
            logits_list = [model(images)]
            if use_tta:
                logits_list.append(model(torch.flip(images, dims=[-1])))      # h-flip
                logits_list.append(model(torch.flip(images, dims=[-2])))      # v-flip
                logits_list.append(model(torch.flip(images, dims=[-2, -1])))  # both
            probs = torch.stack([torch.softmax(l, dim=1) for l in logits_list]).mean(0)
            all_probs.append(probs.cpu())
            all_ids.extend(image_ids.numpy().tolist())
    return torch.cat(all_probs, dim=0).numpy(), np.array(all_ids)

## 8. 5-Fold Training

In [ ]:
N_SPLITS = 5

EXPERIMENTS = [
    {'name': 'se_resnet34_scratch', 'epochs': 200, 'lr': 0.030, 'weight_decay': 5e-4, 'patience': 50},
    {'name': 'densenet_bc100',      'epochs': 200, 'lr': 0.030, 'weight_decay': 5e-4, 'patience': 50},
    {'name': 'scatter_resnet34',    'epochs': 200, 'lr': 0.030, 'weight_decay': 5e-4, 'patience': 50},
]

skf              = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
all_fold_results = []
all_histories    = {}
all_model_probs  = {}
ids_reference    = None


def make_loaders(tr_df, va_df):
    tr = TildaDataset(tr_df, transform=train_tfms, has_labels=True)
    va = TildaDataset(va_df, transform=eval_tfms,  has_labels=True)
    return (
        DataLoader(tr, batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS, pin_memory=True),
        DataLoader(va, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True),
    )


for exp in EXPERIMENTS:
    model_name       = exp['name']
    model_fold_probs = []

    print('\n' + '#' * 80)
    print(f'5-FOLD TRAINING: {model_name}')
    print('#' * 80)

    for fold, (tr_idx, va_idx) in enumerate(skf.split(df, df['label']), start=1):
        fold_name = f'{model_name}_fold{fold}'
        print('\n' + '=' * 70)
        print(f'  {fold_name}')
        print('=' * 70)

        tr_ld, va_ld = make_loaders(
            df.iloc[tr_idx].reset_index(drop=True),
            df.iloc[va_idx].reset_index(drop=True),
        )

        model = build_model(model_name).to(DEVICE)
        history, best_path, best_acc, best_epoch, elapsed = train_model(
            model, tr_ld, va_ld, fold_name,
            epochs=exp['epochs'], lr=exp['lr'],
            weight_decay=exp['weight_decay'], patience=exp['patience'],
        )

        history.to_csv(OUTPUT_DIR / f'history_{fold_name}.csv', index=False)
        all_histories[fold_name] = history

        ckpt = torch.load(best_path, map_location=DEVICE)
        model.load_state_dict(ckpt['model_state_dict'])
        probs, ids = predict_proba(model, test_loader, use_tta=True)

        if ids_reference is None:
            ids_reference = ids
        else:
            assert np.array_equal(ids_reference, ids)

        model_fold_probs.append(probs)
        fold_sub = pd.DataFrame({'id': ids, 'label': probs.argmax(1)}).sort_values('id')
        fold_sub.to_csv(SUBMISSION_DIR / f'submission_{fold_name}_tta_labels0.csv', index=False)

        all_fold_results.append({'model': model_name, 'fold': fold, 'fold_name': fold_name,
                                  'best_val_acc': best_acc, 'best_epoch': best_epoch,
                                  'training_time_min': elapsed, 'submission_labels': '0..7'})
        del model
        if DEVICE.type == 'cuda': torch.cuda.empty_cache()

    model_probs = np.mean(model_fold_probs, axis=0)
    all_model_probs[model_name] = model_probs
    model_sub = pd.DataFrame({'id': ids_reference, 'label': model_probs.argmax(1)}).sort_values('id')
    sub_path  = SUBMISSION_DIR / f'submission_{model_name}_5fold_tta_labels0.csv'
    model_sub.to_csv(sub_path, index=False)
    print(f'Saved: {sub_path.name}')
    print(model_sub['label'].value_counts().sort_index())

fold_results_df = pd.DataFrame(all_fold_results)
fold_results_df.to_csv(OUTPUT_DIR / 'model_results_advanced.csv', index=False)
fold_results_df.sort_values(['model', 'fold'])


################################################################################
5-FOLD TRAINING: se_resnet34_scratch
################################################################################

  se_resnet34_scratch_fold1


  0%|          | 0/59 [00:00<?, ?it/s]

  0%|          | 0/15 [00:00<?, ?it/s]

se_resnet34_scratch_fold1 | ep 001 | train 0.1727/2.1621 | val 0.1712/2.0706 | best 0.1712 @ 1


  0%|          | 0/59 [00:00<?, ?it/s]

...

se_resnet34_scratch_fold1 | ep 143 | train 0.6785/0.9464 | val 0.6638/1.3404 | best 0.6638 @ 143


  0%|          | 0/59 [00:00<?, ?it/s]

  0%|          | 0/15 [00:00<?, ?it/s]

se_resnet34_scratch_fold1 | ep 144 | train 0.6806/1.0553 | val 0.6131/1.3285 | best 0.6638 @ 143


  0%|          | 0/59 [00:00<?, ?it/s]

## 9. Diagnostics

In [ ]:
display(
    fold_results_df.groupby('model')['best_val_acc']
    .agg(['mean', 'std', 'min', 'max'])
    .sort_values('mean', ascending=False)
)

fig, ax = plt.subplots(figsize=(12, 5))
for fold_name, hist in all_histories.items():
    ax.plot(hist['epoch'], hist['val_acc'], label=fold_name, alpha=0.75)
ax.set_xlabel('Epoch')
ax.set_ylabel('Validation accuracy')
ax.grid(True)
ax.legend(fontsize=7)
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'val_accuracy_advanced.png', dpi=150)
plt.show()

## 10. Global Ensemble Submission

SE-ResNet34 × 5 + DenseNet-BC-100 × 5 + ScatteringResNet34 × 5 = 15 predictions averaged.

In [ ]:
available = list(all_model_probs.keys())
print('Models in ensemble:', available)

ens_probs = np.mean([all_model_probs[n] for n in available], axis=0)
ens_sub   = pd.DataFrame({'id': ids_reference, 'label': ens_probs.argmax(1)}).sort_values('id')
ens_path  = SUBMISSION_DIR / 'submission_advanced_ensemble_tta_labels0.csv'
ens_sub.to_csv(ens_path, index=False)
print('Saved:', ens_path.name)
print(ens_sub['label'].value_counts().sort_index())
display(ens_sub.head())